# Modelling: search and ranking on OTTO sessions

Follows `_doc/modeling_guide.md`, one section per step, decisions in `modeling/model_log.md`, every experiment in `modeling/experiments.csv`, one issue per step (umbrella #110). It starts from the EDA deliverables (`eda/eda_summary.md`, `eda/preprocessing_spec.md`). Outputs are not committed with the notebook (nbstripout), so every step carries its reasoning in text cells: the candidates considered, why the step exists, what the result means.

Conventions used throughout: unit = one query (a session prefix plus its 200 candidate items); primary metric NDCG@10 of the ranking produced inside the pool, ideal DCG from the full label set; folds are made over queries; seed 42; a difference smaller than the noise threshold of Step 0 is a tie and the simpler model wins.

In [ ]:
import importlib.metadata as md_
import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "notebooks"))
from model_cv import (
    POOL,
    cv,
    experiment_rows,
    fit_score,
    folds,
    lgbm,
    linear,
    load_dev,
    ndcg10,
    pool_ceiling,
    summarise,
)
from model_data import DEV_TABLE, HOLDOUT_TABLE, build_holdout, holdout_fingerprint
from model_utils import SEED, cached, log_experiments, log_step, save_table
from sklearn.metrics import average_precision_score, roc_auc_score
from split import DAY_MS, START_MS

VERSIONS = {p: md_.version(p) for p in ("polars", "numpy", "scikit-learn", "lightgbm", "scipy")}
print("python", platform.python_version(), VERSIONS, "| seed", SEED)

## Step 0. Setup and reference points
*Kind: diagnostic. Issue #111.* Read the inputs, settle the holdout, rebuild the EDA preprocessing as one pipeline, and set the reference score, the noise threshold and the success criterion that every later step is judged against.

#### Why this step exists here
Every later number is only meaningful against a reference, a noise level and an untouched test set. This step also checks that the EDA pipeline can be rebuilt and reproduces the EDA baselines; if it could not, nothing built on it could be trusted.

#### Options considered
- **Holdout:** keep the EDA holdout (sessions starting on days 14-15), build a fresh window from unused days (16-17), or use a later window (days 21-22 or 26-27). The first was rejected because earlier issues scored five methods on it; the third because the weekly-arrival demonstrations of the MLOps phase need those windows.
- **Baselines:** the rule already in use (the pool order from reciprocal-rank fusion), random order, single-feature rankings (co-visitation rank, popularity rank), regularised logistic regression, LightGBM at defaults as a classifier and as a lambdarank ranker.
- **Noise floor:** five fold seeds x three folds on 10,000 queries; the guide's threshold is the larger of the fold standard deviation and the seed spread of the reference; the paired difference between options on the same folds is reported next to it.
- **Ceiling:** a perfect reordering of the pool, the benchmarks of the earlier phase (reference only, different queries), label noise from the EDA.

In [ ]:
# A. inputs from the EDA phase, and what is assumed where the guide asks for facts we were not given
required = ["eda/eda_summary.md", "eda/eda_log.md", "eda/preprocessing_spec.md", "eda/tables/step17_excluded_columns.csv", "eda/tables/step16_baselines.csv", "data/features/clean_dev.parquet"]
missing = [p for p in required if not (ROOT / p).exists()]
assert not missing, f"missing inputs: {missing}"
print("all EDA inputs present:", ", ".join(required))
assumptions = pl.DataFrame([
    {"item": "primary metric", "value": "NDCG@10 inside the 200-candidate pool, ideal DCG from the full label set, mean over queries", "status": "assumed (from AGENTS.md and issues #13/#17; the EDA used PR-AUC only for speed)"},
    {"item": "error costs", "value": "unknown; a ranking has no fixed cut-off, the operating point is the top-10 result list", "status": "assumed; no cost-based threshold is invented (Step 5)"},
    {"item": "latency limit", "value": "under 100 ms per query (200 candidates) on one Lambda CPU, p95", "status": "assumed (serverless architecture, low traffic)"},
    {"item": "model size limit", "value": "under 50 MB for the frozen artifact", "status": "assumed"},
    {"item": "interpretability", "value": "feature importance and a model card are enough; no per-decision explanation required", "status": "assumed"},
    {"item": "prohibited features", "value": "none; identifiers (session, aid) are not used as features (EDA Steps 6, 13)", "status": "assumed"},
    {"item": "compute budget", "value": "one 8 GB laptop CPU, no cloud spend, hours not days; screening on 10,000 queries, finalists on the 40,000", "status": "assumed"},
    {"item": "retraining frequency", "value": "weekly (matches the weekly data-arrival plan of the MLOps phase)", "status": "assumed"},
    {"item": "sensitive attributes", "value": "none in the data; disparity analysis in Step 6 is limited to the synthetic categories and prefix length", "status": "assumed"},
])
save_table("step0_assumptions", assumptions); print(assumptions.select("item", "value", "status"))

In [ ]:
# B. the holdout. The EDA said days 14-15 were never read by the EDA notebook; but earlier phases scored five methods on that window.
cmp = json.loads((ROOT / "data" / "results" / "comparison.json").read_text())
spent = [(r["method"], r["n"], r["ndcg10"]) for r in cmp["full_window"]]
print("window of days 14-15 (features as of day 14) was already used to compare methods in issues #17-#29:")
for m, n, v in spent: print(f"   {m}: NDCG@10 {v} on {n:,} queries")
print("-> it cannot be an untouched holdout (rule 1). A fresh window is built from days that no earlier issue used.")
fp = build_holdout()
fp2 = holdout_fingerprint(); assert fp["sha256_of_sorted_session_ids"] == fp2["sha256_of_sorted_session_ids"]
dev_sessions = set(pl.scan_parquet(DEV_TABLE).select("session").unique().collect()["session"].to_list())
w0_sessions = set(pl.scan_parquet(ROOT / "data" / "features" / "eval_w0.parquet").select("session").unique().collect()["session"].to_list())
h0_sessions = set(pl.scan_parquet(HOLDOUT_TABLE).select("session").unique().collect()["session"].to_list())
assert not (h0_sessions & dev_sessions) and not (h0_sessions & w0_sessions), "the fresh holdout shares sessions with development or the used window"
print("fresh holdout:", fp)
print(f"no session is shared with development ({len(dev_sessions):,} queries) or with the used window ({len(w0_sessions):,}); the holdout table is written but has not been scored")
dev_fp = {"queries": len(dev_sessions), "rows": len(dev_sessions) * POOL}

In [ ]:
# C. rebuild the EDA pipeline and reproduce the EDA baselines (Step 16 of the EDA: time-based split, 3 samples of 6,000 queries)
first = pl.scan_parquet(ROOT / "data" / "sample" / "events.parquet").group_by("session").agg(pl.col("ts").min().alias("first_ts")).collect()
def days_of(sessions):
    return pl.DataFrame({"session": sessions}).join(first, on="session", how="left")["first_ts"].to_numpy().__sub__(START_MS) // DAY_MS
def time_split(q):
    d = days_of(q.sessions); return np.flatnonzero(d <= 12), np.flatnonzero(d == 13)
rows = []
for s in (42, 43, 44):
    q = load_dev(6000, seed=s); tr, va = time_split(q); q_tr, q_va = q.take(tr), q.take(va)
    for spec in (linear(), lgbm()):
        sc, *_ = fit_score(spec, q_tr, q_va)
        rows.append({"option": spec.name, "sample": s, "pr_auc": average_precision_score(q_va.y, sc.ravel()), "ndcg10": float(ndcg10(sc, q_va.grade, q_va.idcg).mean())})
rep = pl.DataFrame(rows).group_by("option", maintain_order=True).agg(pl.col("pr_auc").mean(), pl.col("pr_auc").std().alias("pr_auc_sd"), pl.col("ndcg10").mean())
eda = pl.read_csv(ROOT / "eda" / "tables" / "step16_baselines.csv").filter(pl.col("option").str.starts_with("B.") | pl.col("option").str.starts_with("C."))
eda_pr, eda_sd = eda["pr_auc"].to_list(), eda["pr_auc_sd"].to_list()
rep = rep.with_columns(pl.Series("eda_pr_auc", eda_pr), pl.Series("eda_spread", eda_sd)).with_columns(((pl.col("pr_auc") - pl.col("eda_pr_auc")).abs() <= pl.max_horizontal("pr_auc_sd", "eda_spread")).alias("reproduced"))
save_table("step0_reproduce_eda", rep); print(rep)
assert rep["reproduced"].all(), "the rebuilt pipeline does not reproduce the EDA baselines: stop and find out why"

In [ ]:
# D. baselines on the screening sample (10,000 queries) with five fold seeds: the noise floor
dev10 = load_dev(10_000, seed=SEED); SEEDS5 = (42, 43, 44, 45, 46)
print(f"screening sample: {dev10.nq:,} queries, {dev10.df.height:,} rows, {int(dev10.y.sum()):,} positives")
def heuristic(name, fn):
    out = []
    for seed in SEEDS5:
        for fold, (_, va) in enumerate(folds(dev10.nq, seed)):
            qv = dev10.take(va); s = fn(qv)
            out.append({"option": name, "seed": seed, "fold": fold, "ndcg10": float(ndcg10(s, qv.grade, qv.idcg).mean()), "pr_auc": average_precision_score(qv.y, s.ravel()), "roc_auc": roc_auc_score(qv.y, s.ravel()),
                        "fit_seconds": 0.0, "predict_seconds": 0.0, "size_mb": 0.0, "n_features": 0, "queries": qv.nq})
    return out
pos = -np.arange(POOL, dtype=np.float64)[None, :]
HEUR = {"pool order (rule in use: reciprocal-rank fusion)": lambda q: np.repeat(pos, q.nq, 0),
        "random within the pool": lambda q: np.random.default_rng(SEED).random((q.nq, POOL)),
        "co-visitation rank": lambda q: -q.df["rank_covis"].to_numpy().reshape(q.nq, POOL) + 1e-6 * np.repeat(pos, q.nq, 0),
        "popularity rank": lambda q: -q.df["rank_pop"].to_numpy().reshape(q.nq, POOL) + 1e-6 * np.repeat(pos, q.nq, 0)}
def run0():
    rows = [r for n, fn in HEUR.items() for r in heuristic(n, fn)]
    for spec in (linear(), lgbm(), lgbm(name="LightGBM lambdarank defaults", rank=True)):
        t0 = time.time(); r, _ = cv(spec, dev10, seeds=SEEDS5); rows += r.to_dicts(); print(f"  {spec.name}: {time.time() - t0:.0f} s")
    return pl.DataFrame(rows)
cv0 = cached("step0_baselines_rows", run0); sum0 = summarise(cv0); save_table("step0_baselines", sum0)
print(sum0.select("option", "ndcg10", "ndcg10_sd", "seed_spread", "pr_auc", "roc_auc", "fit_seconds", "size_mb").sort("ndcg10", descending=True))

In [ ]:
# E. noise floor, ceiling, reference and success criterion
ref_name = sum0.sort("ndcg10", descending=True)["option"][0]; ref = sum0.filter(pl.col("option") == ref_name).row(0, named=True)
noise = max(ref["ndcg10_sd"], ref["seed_spread"])
# paired view: how much of the fold-to-fold variation is shared between options (same folds)?
pv = cv0.select("option", "seed", "fold", "ndcg10"); refv = pv.filter(pl.col("option") == ref_name).rename({"ndcg10": "ref"}).drop("option")
paired = pv.join(refv, on=["seed", "fold"]).with_columns((pl.col("ndcg10") - pl.col("ref")).alias("d")).group_by("option").agg(pl.col("d").mean().alias("mean_diff_to_reference"), pl.col("d").std().alias("paired_sd")).sort("mean_diff_to_reference", descending=True)
save_table("step0_paired_to_reference", paired); print("paired difference to the reference (same folds):"); print(paired)
ceil = pool_ceiling(dev10.grade, dev10.idcg); success = 2 * noise
share_zero = float((ceil == 0).mean())
print(f"reference: {ref_name} NDCG@10 {ref['ndcg10']:.4f}; fold sd {ref['ndcg10_sd']:.4f}, seed spread {ref['seed_spread']:.4f} -> noise threshold {noise:.4f}; success criterion (2 x noise) {success:.4f}")
print(f"ceiling: a perfect reordering of the 200-candidate pool reaches NDCG@10 {ceil.mean():.4f}; {share_zero:.1%} of queries have no relevant item in their pool (nothing to find)")
print("benchmarks on the spent window (reference only, different queries): co-visitation + fill 0.2144, LightGBM lambdarank 0.2153, DCN-V2 + MMoE 0.2069 (data/results/comparison.json)")
print("label noise: 14 of 8,000,000 rows have conflicting labels (EDA Step 4), negligible")
ceiling_tbl = pl.DataFrame([{"reference": ref_name, "ndcg10": ref["ndcg10"], "noise_threshold": noise, "success_criterion": success, "pool_ceiling": float(ceil.mean()), "queries_without_reachable_item": share_zero}])
save_table("step0_reference", ceiling_tbl)
desc = {n: n for n in sum0["option"].to_list()}
print(log_experiments(experiment_rows(0, cv0, sum0, desc, "15 EDA features" if False else "F15", {}, SEEDS5)), "experiments logged")

In [ ]:
top = sum0.sort("ndcg10", descending=True); best_learned = sum0.filter(pl.col("option").str.contains("regularised|LightGBM")).sort("ndcg10", descending=True).row(0, named=True)
e0 = log_step(0, "Setup and reference points", "Diagnostic",
    "Inputs and assumptions listed; holdout status checked and a fresh holdout built; EDA baselines reproduced with the rebuilt pipeline; baselines on the 10,000-query screening sample with five fold seeds: pool order (the rule already in use), random, co-visitation rank, popularity rank, regularised linear, LightGBM defaults (classifier, class weights), LightGBM lambdarank defaults; ceiling = perfect reordering of the pool.",
    f"Holdout: days 14-15 were already used to score five methods in issues #17-#29 (best NDCG@10 0.2153), so they are treated as spent; a fresh holdout of {fp['queries']:,} queries ({fp['rows']:,} rows; sessions starting on days 16-17, features as of day 16) was built and fingerprinted (sha256 of sorted session ids {fp['sha256_of_sorted_session_ids'][:12]}...), disjoint from development and from the used window, and not scored. Reproduction: the rebuilt pipeline gives PR-AUC " + ", ".join(f"{r['option']} {r['pr_auc']:.4f} (EDA {r['eda_pr_auc']:.4f})" for r in rep.iter_rows(named=True)) + ". Baseline NDCG@10 on the screening sample (mean over 15 folds): " + "; ".join(f"{r['option']} {r['ndcg10']:.4f}" for r in top.iter_rows(named=True)) + f". Reference: {ref_name} {ref['ndcg10']:.4f}; fold sd {ref['ndcg10_sd']:.4f}, seed spread {ref['seed_spread']:.4f}, noise threshold {noise:.4f}, success criterion {success:.4f}. Paired difference to the reference on the same folds: " + "; ".join(f"{r['option']} {r['mean_diff_to_reference']:+.4f} (paired sd {r['paired_sd']:.4f})" for r in paired.iter_rows(named=True) if r["option"] != ref_name) + f". Ceiling: {ceil.mean():.4f} (a perfect pool reordering); {share_zero:.0%} of queries have no relevant item in the pool. No learned baseline beats the reference: the reference is one feature (the co-visitation rank).",
    f"Reference score = {ref_name} at {ref['ndcg10']:.4f}. Noise threshold {noise:.4f} (larger of fold standard deviation and seed spread of the reference). Success criterion {success:.4f} NDCG@10 over the reference (2 x noise; no business figure was given). Holdout for the modelling phase: the fresh window (days 16-17), scored once in Step 8; the days 14-15 window is spent and only a reference.",
    "Rule 1 of the guide (holdout used once) cannot hold for days 14-15, so a new window from unused days was built with the same code as issue #22. The noise threshold follows the guide (larger of fold sd and seed spread of the reference). The constraints the guide asked for were not given, so they are assumed and marked (table above).",
    "Keeping days 14-15 as the holdout: rejected, it was used to compare methods. Using a later window that the MLOps plan needs (days 21-22, 26-27): rejected, it would spend windows meant for the weekly-arrival demonstrations. Rebuilding the pipeline by a different route: not needed, the EDA baselines were reproduced within their spread.",
    "High for the holdout decision and the reproduction; Medium for the noise threshold (three folds of 3,333 queries; the paired differences are much tighter than this threshold, see the paired table).",
    "All later steps compare against the reference and the noise threshold; if nothing beats the reference by more than the noise threshold after Steps 1 to 3 the guide says to stop and ask the human. Step 8 scores the fresh holdout once. Step 6 must flag the query-category features (the query category comes from the held-out click, known leak of issue #10).")
print(e0)

#### Chosen approach and rationale
**Decision.** Reference score = co-visitation rank at 0.1815. Noise threshold 0.0082 (larger of fold standard deviation and seed spread of the reference). Success criterion 0.0164 NDCG@10 over the reference (2 x noise; no business figure was given). Holdout for the modelling phase: the fresh window (days 16-17), scored once in Step 8; the days 14-15 window is spent and only a reference.

**Why.** Rule 1 of the guide (holdout used once) cannot hold for days 14-15, so a new window from unused days was built with the same code as issue #22. The noise threshold follows the guide (larger of fold sd and seed spread of the reference). The constraints the guide asked for were not given, so they are assumed and marked (table above).

**Rejected.** Keeping days 14-15 as the holdout: rejected, it was used to compare methods. Using a later window that the MLOps plan needs (days 21-22, 26-27): rejected, it would spend windows meant for the weekly-arrival demonstrations. Rebuilding the pipeline by a different route: not needed, the EDA baselines were reproduced within their spread.

**Confidence.** High for the holdout decision and the reproduction; Medium for the noise threshold (three folds of 3,333 queries; the paired differences are much tighter than this threshold, see the paired table).

#### Interpretation: what we understand from the results
Three findings shape everything that follows. First, the EDA statement that the holdout was never read was true of the EDA notebook only: days 14-15 had been used to compare five methods in the first phase, so it was replaced by a fresh window (43,782 queries, days 16-17) that is fingerprinted and will be scored once in Step 8. Second, the rebuilt pipeline reproduces the EDA baselines (PR-AUC 0.0579 and 0.0592 against 0.0581 and 0.0597). Third, and most important: no learned baseline beats the simplest reference. Sorting the pool by the co-visitation rank alone gives NDCG@10 0.1815; regularised regression, LightGBM and lambdarank at defaults score 0.1798, 0.1799 and 0.1787 (paired differences -0.0018, -0.0017 and -0.0028, paired sd about 0.002). They do beat it on PR-AUC (0.060-0.065 against 0.057), so they order the whole pool better, but not the top ten, which is what NDCG@10 rewards; this is the same picture as the EDA (models add +0.008 to +0.010 PR-AUC over the co-visitation rank). The rule already in use, the fused pool order, scores only 0.128, because the popularity ranks it blends in are weak (popularity rank alone 0.040, random 0.0135). The guide's noise threshold is 0.0082, set by the fold-to-fold difficulty of three folds of 3,333 queries; the paired differences between options are five times tighter, so the threshold is conservative and is used for ties, while key comparisons also get a paired bootstrap interval. The ceiling is 0.566 for a perfect reordering, but 38% of queries have no relevant item in their pool at all: the pool, not the ranker, limits the score. The success criterion (twice the noise threshold) is 0.0164 over the reference; if Steps 1 to 3 cannot reach it the guide requires asking the human. One caveat carried forward: the query category comes from the held-out click (known leak, issue #10), so absolute scores overstate what real search would give.

#### Impact on next steps
All later steps compare against the reference and the noise threshold; Step 8 scores the fresh holdout once. Step 6 must flag the query-category features (the query category comes from the held-out click, known leak of issue #10).

#### Out of scope
- A business figure for the smallest useful uplift: none given, so twice the noise threshold is used (assumed).
- Raising the share of queries with a reachable item (38% have none): candidate generation, not modelling.
- Confirmation of the assumed constraints (latency, size, weekly retraining): listed for review in Step 9.